# GraphRAG: trace service dependencies with provenance

## Northstar Cloud incident scenario

Checkout conversion drops after a payments deployment. A responder needs to connect **deployment → service → dependency → region → owner** with evidence, not merely retrieve chunks that mention similar words. This notebook builds a bounded, tenant-aware graph retrieval controller without a database or model API.


## Learning objectives

- Model entities, typed relations, provenance, and tenant metadata.
- Resolve seed entities and retrieve a bounded local neighborhood.
- Find explainable relationship paths.
- Preserve citations while linearizing facts for an LLM.
- Test cycles, missing entities, tenant isolation, and high-degree nodes.

Read the [GraphRAG lesson](README.md) alongside this notebook. The primary research reference is [From Local to Global](https://arxiv.org/abs/2404.16130).


## Local retrieval lifecycle

```text
[Question + user identity]
            |
            v
[Resolve seed entities] --> [Apply tenant + confidence policy]
                                      |
                                      v
                    [Bounded neighborhood / path retrieval]
                                      |
                                      v
                         [Fact context + provenance]
                                      |
                                      v
                         [Verify claims and citations]
                           | supported             | unsupported
                           v                       v
                     [Grounded answer]       [Abstain / clarify]
```

A GraphRAG controller makes entity resolution, authorization, depth, fan-out, result limits, and truncation visible. A graph is not permission to explore indefinitely.


## 1 — Build a small, sourced property graph

Each `Fact` is a typed edge plus source, tenant, confidence, and revision. In a production graph, retain the source span and extraction run as well.


In [ ]:
from examples.advanced.graph_rag import EntityGraph, Fact, GraphPolicy, linearize

facts = [
    Fact("f1", "deploy-842", "changed", "payments-api", "deployments/842.json", revision="2026-08-09"),
    Fact("f2", "payments-api", "depends_on", "token-service", "services/payments.md", revision="2026-08-01"),
    Fact("f3", "payments-api", "serves", "Europe", "services/payments.md", revision="2026-08-01"),
    Fact("f4", "payments-api", "owned_by", "Payments Team", "services/owners.md", revision="2026-08-01"),
    Fact("f5", "token-service", "has_incident", "rotation-latency", "incidents/551.md", revision="2026-08-09"),
    Fact("f6", "payments-api", "owned_by", "Other Tenant Team", "other/owners.md", tenant="other-tenant"),
]
graph = EntityGraph(facts)


## 2 — Local graph retrieval

For relationship questions, resolve named entities first, then traverse only enough hops to prove the requested connection. This is different from global GraphRAG: global questions such as “what themes explain all Q3 incidents?” require community reports or another aggregation strategy.


In [ ]:
policy = GraphPolicy(max_hops=2, max_facts=8, permitted_tenants=frozenset({"northstar"}))
evidence = graph.retrieve("What changed around payments-api and token-service?", policy)
print(evidence.seed_entities, evidence.reason, evidence.truncated)
print(linearize(evidence))
assert "f6" not in {fact.fact_id for fact in evidence.facts}
assert {"f1", "f2", "f3", "f4", "f5"}.issubset({fact.fact_id for fact in evidence.facts})


### Why authorization happens before traversal

Filtering only final results is unsafe: intermediate edges can reveal the existence of a tenant, project, incident, or relationship. The graph policy is evaluated at every traversal step. In production, enforce the same filters in the graph query itself and in any source-span lookup.


## 3 — Retrieve an explainable path

A path is especially useful when the answer must explain *why* two entities are connected. A response should cite each edge, not claim a relationship from graph proximity alone.


In [ ]:
paths = graph.paths("deploy-842", "token-service", policy)
for path in paths:
    print(" -> ".join(f"{fact.subject} -[{fact.relation}]-> {fact.object}" for fact in path))
assert len(paths) == 1
assert [fact.fact_id for fact in paths[0]] == ["f1", "f2"]


## 7 — Bound fan-out and make truncation visible

High-degree hubs can flood a prompt with irrelevant facts. A production system needs both a hop budget and a fact/fan-out budget, and it must report that the result was truncated rather than pretending it saw the whole graph.


In [ ]:
hub_facts = facts + [Fact(f"hub-{i}", "payments-api", "integrates_with", f"partner-{i}", "partners.md") for i in range(20)]
hub_graph = EntityGraph(hub_facts)
limited = hub_graph.retrieve("payments-api", GraphPolicy(max_hops=1, max_facts=4))
print(len(limited.facts), limited.truncated, [fact.fact_id for fact in limited.facts])
assert len(limited.facts) == 4


## Visualize the incident graph

The topology below is a portable SVG, so it renders directly on GitHub and in most notebook viewers. It shows the exact facts used in the local incident investigation; labels are typed relations, not vague associations.

![Northstar incident graph](graph_topology.svg)

The deployment-to-token-service path is a two-hop proof. The region and owner branches are additional facts that can support impact and escalation recommendations, but should not be added to context if the question only asks about a component dependency.


## 4 — Missing entities and ambiguous questions

A graph cannot answer a relationship question until it resolves safe seed entities. Treat a failed resolution as a useful result: ask for a canonical service, incident, or deployment ID rather than guessing an alias.


In [ ]:
missing = graph.retrieve("Which team owns catalog-api?", policy)
print(missing.seed_entities, missing.reason, missing.facts)
assert missing.reason == "no-resolved-entity"
assert missing.facts == ()


## 5 — Prove tenant isolation with an adversarial fixture

`f6` has the same service subject but belongs to another tenant. It must never become a model-visible fact merely because it is one hop away from an authorized node.


In [ ]:
isolated = graph.retrieve("Who owns payments-api?", policy)
print(linearize(isolated))
assert "Other Tenant Team" not in linearize(isolated)
assert "f6" not in {fact.fact_id for fact in isolated.facts}
print("Tenant boundary preserved; citations:", isolated.citations())


## 6 — Add supporting text to graph paths

Graph facts establish relationships; source passages supply operational detail. A production answer should pair the shortest relevant path with the source spans that justify each edge. This prevents a model from treating a graph edge as a license to invent a remediation step.


In [ ]:
source_spans = {
    "f1": "Deployment 842 updated payments-api at 08:42 UTC.",
    "f2": "payments-api calls token-service for credential validation.",
}
path = paths[0]
context = [
    {"fact_id": fact.fact_id, "edge": f"{fact.subject} {fact.relation} {fact.object}", "source": fact.source, "span": source_spans.get(fact.fact_id)}
    for fact in path
]
for item in context:
    print(item)
assert all(item["span"] for item in context)


## 8 — Compare local graph, global graph, and hybrid retrieval

| Mode | Best question | Context | Main risk |
| --- | --- | --- | --- |
| Local | “How is deploy-842 connected to Europe?” | Path/neighborhood + source spans | Entity resolution errors and fan-out |
| Global | “What patterns recur across incidents?” | Community reports / map-reduce summaries | Expensive indexing and stale summaries |
| Hybrid | “What changed and which customers are affected?” | Text seeds + graph path + passages | Inconsistent IDs across stores |

Use text/vector retrieval when a passage is enough. Add graph retrieval only where relational structure measurably improves answer quality or explainability.


## 9 — Evaluation and production readiness

Evaluate extraction precision/recall, entity-resolution merge/split errors, seed/path recall, authorized recall, path faithfulness, citation correctness, p95 latency, graph freshness, and cross-tenant traversal attempts.

Production checklist:

- Version every fact and preserve source spans, extraction model, schema, and timestamp.
- Validate relation vocabulary and canonical entity IDs before writes.
- Apply identity filters in graph queries, source retrieval, caches, and traces.
- Bound hops, fan-out, result count, and query time; expose truncation.
- Offer text retrieval or abstention when the graph is stale/unavailable.
- Keep generated graph-query access constrained; never permit arbitrary model-generated Cypher.

Explore [Microsoft GraphRAG](https://microsoft.github.io/graphrag/), [Neo4j GraphRAG for Python](https://neo4j.com/docs/neo4j-graphrag-python/current/index.html), and [LlamaIndex PropertyGraphIndex](https://llamaindex.openml.io/python/framework/module_guides/indexing/lpg_index_guide/) after mastering this deterministic boundary.


## Exercises

1. Add an alias resolver for “payments API” and measure false merges.
2. Add a stale fact and build a freshness policy that refuses it for current incident response.
3. Add a two-tenant graph and prove every path stays tenant-scoped.
4. Add supporting source spans and require every answer claim to cite a fact ID and span.
5. Create ten relationship questions and compare GraphRAG against chunk-only retrieval. Which task types justify graph indexing?
6. Design a global community-report workflow for quarterly incident themes, including refresh and evaluation policy.
